<a href="https://colab.research.google.com/github/onishchenkoanastasia5-glitch/Machine_learning/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9E%D0%BD%D0%B8%D1%89%D0%B5%D0%BD%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Онищенко Анастасія  
**Група:** ФІТ 3-12  
**Дата:** 24.11

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідження набору даних
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())
print('Кількість унікальних фільмів:     ', ratings_df['item_id'].nunique())
print('Кількість оцінок:                 ', len(ratings_df))

print('\nСтатистика rating:')
print(ratings_df['rating'].agg(['min', 'max', 'mean']).round(3))

print('\nПропущені значення:')
print(ratings_df.isna().sum())

print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
sparsity = 1 - len(ratings_df) / (n_users * n_items)
print(f'\nРозрідженість матриці «користувач–фільм»: {sparsity:.2%}')

Кількість унікальних користувачів: 943
Кількість унікальних фільмів:      1682
Кількість оцінок:                  100000

Статистика rating:
min     1.00
max     5.00
mean    3.53
Name: rating, dtype: float64

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Розрідженість матриці «користувач–фільм»: 93.70%


**Короткі спостереження щодо даних:**

У наборі `ml-100k` є 943 користувачів, 1682 фільмів і 100000 оцінок. Пропущених значень немає. Найчастіше трапляються оцінка 4, тобто розподіл зміщений у бік високих оцінок. Матриця дуже розріджена (≈ 93.70% порожніх клітинок), тому для рекомендацій важливо, щоб модель добре працювала з малою кількістю оцінок на користувача й фільм.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Розбиття 80% / 20%
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

print('train_df:', train_df.shape)
print('test_df: ', test_df.shape)

train_df: (80000, 4)
test_df:  (20000, 4)


In [6]:
# TODO 3.2 — Об’єкти для Surprise
reader = Reader(rating_scale=data.reader.rating_scale)

# Тільки навчальна частина — для GridSearchCV та навчання
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)

# Тестова вибірка — список кортежів (user_id, item_id, rating)
testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print('Шкала оцінок:', reader.rating_scale)
print('Розмір train:', len(train_df), '| Розмір test:', len(testset))

Шкала оцінок: (1, 5)
Розмір train: 80000 | Розмір test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

**SVD**
- `n_factors` (50, 100) — кількість прихованих факторів. Більше факторів дає виразнішу модель, але підвищує ризик перенавчання на розрідженій матриці.
- `lr_all` (0.005, 0.010) — швидкість навчання SGD. Занадто мала — повільна збіжність за фіксовані 20 епох, занадто велика — нестабільність.
- `reg_all` (0.02, 0.05) — L2-регуляризація. Захищає від перенавчання; на розріджених даних зазвичай корисна трохи більша регуляризація.

**KNNBasic**
- `k` (20, 40) — максимальна кількість сусідів: більше сусідів згладжує прогноз, менше — точніше, але шумніше.
- `min_k` (1, 5) — мінімальна кількість сусідів; якщо їх менше, повертається середнє по всіх оцінках, тож вищий `min_k` робить прогноз стабільнішим.
- `sim_options.name` (`cosine`, `pearson`) — міра схожості; `pearson` враховує різну «щедрість» оцінювання.
- `sim_options.user_based` (`False`/`True`) — схожість між фільмами або між користувачами.

Початкову сітку з шаблону залишено без змін: вона невелика (8 комбінацій для SVD, 16 для KNN × 3 фолди), тому виконується за розумний час. За потреби її можна розширити, наприклад `n_factors: [100, 150]`, `n_epochs: [20, 30]`.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — GridSearchCV для SVD (лише на train_data)
# Додаємо random_state для відтворюваності результатів
param_grid_svd = {**param_grid_svd, 'random_state': [42]}

grid_search_svd = GridSearchCV(
    SVD, param_grid_svd,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('SVD: пошук завершено')

SVD: пошук завершено


In [9]:
# TODO 5.2 — GridSearchCV для KNNBasic (лише на train_data)
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('KNNBasic: пошук завершено')

KNNBasic: пошук завершено


In [10]:
# TODO 5.3 — Результати кросвалідації
for name, gs in [('SVD', grid_search_svd), ('KNNBasic', grid_search_knn)]:
    best_idx = gs.best_index['rmse']
    print(f'=== {name} ===')
    print('Найкращий CV RMSE:', round(gs.best_score['rmse'], 4))
    # MAE для тієї самої комбінації параметрів, що й найкращий RMSE
    print('CV MAE (для цих параметрів):', round(gs.cv_results['mean_test_mae'][best_idx], 4))
    print('Найкращий CV MAE (серед усіх):', round(gs.best_score['mae'], 4))
    print('Параметри за RMSE:', gs.best_params['rmse'])
    print()

=== SVD ===
Найкращий CV RMSE: 0.9427
CV MAE (для цих параметрів): 0.7438
Найкращий CV MAE (серед усіх): 0.7436
Параметри за RMSE: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05, 'random_state': 42}

=== KNNBasic ===
Найкращий CV RMSE: 1.0281
CV MAE (для цих параметрів): 0.8155
Найкращий CV MAE (серед усіх): 0.8155
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}



**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |


## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 — Найкращі параметри за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']
print('SVD:', best_svd_params)
print('KNN:', best_knn_params)

# TODO 6.2 — Повна навчальна вибірка
trainset = train_data.build_full_trainset()
print('Навчальна вибірка:', trainset.n_users, 'користувачів,',
      trainset.n_items, 'фільмів,', trainset.n_ratings, 'оцінок')

# TODO 6.3 — Навчання моделей
best_svd = SVD(**best_svd_params)   # random_state=42 уже входить до параметрів
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)
print('Моделі навчено')

SVD: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05, 'random_state': 42}
KNN: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Навчальна вибірка: 943 користувачів, 1653 фільмів, 80000 оцінок
Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі навчено


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
# TODO 7.1 — Передбачення для тестової вибірки
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Метрики
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae  = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae  = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD:      RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNNBasic: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

SVD:      RMSE=0.9194, MAE=0.7247
KNNBasic: RMSE=1.0120, MAE=0.7984


In [13]:
# TODO 7.3 — Порівняльна таблиця
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE':  [svd_rmse, knn_rmse],
    'MAE':   [svd_mae, knn_mae],
}).sort_values('RMSE').reset_index(drop=True)

display(results.round(4))

,Model,RMSE,MAE
0,SVD,0.9194,0.7247
1,KNNBasic,1.0120,0.7984


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** на тестовій вибірці SVD отримав RMSE = 0.9194 та MAE = 0.7247, KNNBasic — RMSE = 1.0120 та MAE = 0.7984.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Найкраща за CV RMSE модель
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model, best_name = best_svd, 'SVD'
else:
    best_model, best_name = best_knn, 'KNNBasic'
print('Модель для рекомендацій:', best_name)

# назви фільмів із файлу u.item
try:
    import os
    from surprise.builtin_datasets import get_dataset_dir
    items_path = os.path.join(get_dataset_dir(), 'ml-100k', 'u.item')
    movies = pd.read_csv(items_path, sep='|', header=None, encoding='latin-1',
                         usecols=[0, 1], names=['item_id', 'title'])
    movies['item_id'] = movies['item_id'].astype(str)
except Exception as e:
    movies = None
    print('Назви фільмів недоступні:', e)


def recommend_top_n(model, trainset, user_id, n=10):
    """Топ-n неоцінених користувачем фільмів за прогнозованою оцінкою."""
    user_id = str(user_id)

    # TODO 8.2 — користувач має бути в trainset
    try:
        inner_uid = trainset.to_inner_uid(user_id)
    except ValueError:
        print(f'Користувача {user_id} немає в навчальній вибірці (cold start).')
        return None

    # TODO 8.3 — фільми, які він уже оцінив у train
    rated = {trainset.to_raw_iid(inner_iid) for inner_iid, _ in trainset.ur[inner_uid]}
    candidates = [trainset.to_raw_iid(i) for i in trainset.all_items()
                  if trainset.to_raw_iid(i) not in rated]

    # TODO 8.4 — прогноз і топ-n
    preds = [(iid, model.predict(uid=user_id, iid=iid).est) for iid in candidates]
    top = (pd.DataFrame(preds, columns=['item_id', 'predicted_rating'])
             .sort_values('predicted_rating', ascending=False)
             .head(n).reset_index(drop=True))
    print(f'Користувач {user_id} оцінив {len(rated)} фільмів; кандидатів: {len(candidates)}')
    return top


top10 = recommend_top_n(best_model, trainset, user_id, n=10)
if top10 is not None:
    if movies is not None:
        top10 = top10.merge(movies, on='item_id', how='left')[['item_id', 'title', 'predicted_rating']]
    display(top10.round(3))

Модель для рекомендацій: SVD
Назви фільмів недоступні: [Errno 2] No such file or directory: '/root/.surprise_data/ml-100k/u.item'
Користувач 196 оцінив 29 фільмів; кандидатів: 1624


,item_id,predicted_rating
0,318,4.578
1,427,4.510
2,603,4.456
3,1169,4.441
4,661,4.415
5,479,4.391
6,513,4.356
7,357,4.325
8,64,4.322
9,313,4.301


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

**Висновок:**
Рекомендації мають обмеження: модель не може прогнозувати для **нових користувачів і фільмів** без оцінок (cold start), не враховує жанри, час і контекст, а розріджена матриця знижує точність для рідко оцінюваних фільмів. Також рекомендації ґрунтуються лише на історичних оцінках і можуть підсилювати популярні фільми.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

**Ідея алгоритму (Baseline + Item-kNN на залишках):**

1. *Базова оцінка:* `b_ui = μ + b_u + b_i`, де `μ` — середня оцінка, `b_i` та `b_u` — зміщення фільму й користувача з демпфуванням (регуляризацією) для рідких даних.
2. *Залишки:* `e_ui = r_ui − b_ui` для всіх відомих оцінок.
3. *Схожість фільмів:* косинусна схожість векторів залишків із поправкою `n_common / (n_common + shrink)`, щоб не довіряти схожості, порахованій за кількома спільними користувачами.
4. *Прогноз:* `r̂_ui = b_ui + Σ sim_ij · e_uj / Σ |sim_ij|` по `k` найсхожіших фільмах `j`, оцінених користувачем `u`.

Використовується лише `numpy`/`pandas`, без Surprise. Навчання — на `train_df`, оцінювання — на тому самому `test_df`, що й вище, тому результат можна порівняти з SVD та KNNBasic.

In [15]:
class BaselineItemKNN:
    """Власний рекомендаційний алгоритм: baseline + item-based kNN на залишках."""

    def __init__(self, k=40, shrink=25, reg_i=10, reg_u=15, rating_range=(1, 5)):
        self.k, self.shrink = k, shrink
        self.reg_i, self.reg_u = reg_i, reg_u
        self.lo, self.hi = rating_range

    def fit(self, df):
        users = df['user_id'].astype(str).to_numpy()
        items = df['item_id'].astype(str).to_numpy()
        r = df['rating'].to_numpy(dtype=float)

        self.u_index = {u: n for n, u in enumerate(pd.unique(users))}
        self.i_index = {i: n for n, i in enumerate(pd.unique(items))}
        self.i_raw = list(self.i_index)
        ui = np.array([self.u_index[u] for u in users])
        ii = np.array([self.i_index[i] for i in items])
        nu, ni = len(self.u_index), len(self.i_index)

        # 1. Baseline із демпфуванням
        self.mu = r.mean()
        sum_i = np.bincount(ii, weights=r - self.mu, minlength=ni)
        cnt_i = np.bincount(ii, minlength=ni)
        self.b_i = sum_i / (self.reg_i + cnt_i)

        resid_u = r - self.mu - self.b_i[ii]
        sum_u = np.bincount(ui, weights=resid_u, minlength=nu)
        cnt_u = np.bincount(ui, minlength=nu)
        self.b_u = sum_u / (self.reg_u + cnt_u)

        # 2. Матриця залишків (користувачі × фільми), 0 = немає оцінки
        e = r - (self.mu + self.b_u[ui] + self.b_i[ii])
        E = np.zeros((nu, ni))
        E[ui, ii] = e
        M = np.zeros((nu, ni))
        M[ui, ii] = 1.0

        # 3. Схожість фільмів зі зсувом (shrinkage)
        dot = E.T @ E
        norm = np.sqrt((E ** 2).sum(axis=0))
        cos = dot / (np.outer(norm, norm) + 1e-9)
        common = M.T @ M
        self.S = cos * common / (common + self.shrink)
        np.fill_diagonal(self.S, 0.0)

        self.E, self.M = E, M
        return self

    def _baseline(self, u, i):
        return self.mu + self.b_u[u] + self.b_i[i]

    def predict(self, user_id, item_id):
        u = self.u_index.get(str(user_id))
        i = self.i_index.get(str(item_id))
        # cold start: невідомий користувач/фільм
        if u is None and i is None:
            return self.mu
        if u is None:
            return float(np.clip(self.mu + self.b_i[i], self.lo, self.hi))
        if i is None:
            return float(np.clip(self.mu + self.b_u[u], self.lo, self.hi))

        base = self._baseline(u, i)
        rated = np.flatnonzero(self.M[u])
        if rated.size == 0:
            return float(np.clip(base, self.lo, self.hi))
        sims = self.S[i, rated]
        if rated.size > self.k:
            top = np.argpartition(-np.abs(sims), self.k)[:self.k]
            rated, sims = rated[top], sims[top]
        denom = np.abs(sims).sum()
        if denom < 1e-9:
            return float(np.clip(base, self.lo, self.hi))
        pred = base + (sims * self.E[u, rated]).sum() / denom
        return float(np.clip(pred, self.lo, self.hi))

    def recommend(self, user_id, n=10):
        u = self.u_index.get(str(user_id))
        if u is None:
            print(f'Користувача {user_id} немає в навчальних даних (cold start).')
            return None
        unrated = np.flatnonzero(self.M[u] == 0)
        preds = [(self.i_raw[i], self.predict(user_id, self.i_raw[i])) for i in unrated]
        return (pd.DataFrame(preds, columns=['item_id', 'predicted_rating'])
                  .sort_values('predicted_rating', ascending=False)
                  .head(n).reset_index(drop=True))

In [16]:
# Навчання та оцінювання власного алгоритму на тому самому розбитті
own_model = BaselineItemKNN(k=40, shrink=25, rating_range=reader.rating_scale).fit(train_df)

own_pred = np.array([own_model.predict(u, i) for u, i, _ in testset])
true_r = np.array([r for _, _, r in testset], dtype=float)

own_rmse = float(np.sqrt(np.mean((own_pred - true_r) ** 2)))
own_mae = float(np.mean(np.abs(own_pred - true_r)))

results_all = pd.concat([
    results,
    pd.DataFrame({'Model': ['Власний (Baseline+ItemKNN)'], 'RMSE': [own_rmse], 'MAE': [own_mae]})
]).sort_values('RMSE').reset_index(drop=True)
display(results_all.round(4))

,Model,RMSE,MAE
0,Власний (Baseline+ItemKNN),0.9123,0.7160
1,SVD,0.9194,0.7247
2,KNNBasic,1.0120,0.7984


In [17]:
# Невеликий підбір параметрів власного алгоритму.
tr_part, val_part = sklearn_train_test_split(train_df, test_size=0.15, random_state=42)
val_true = val_part['rating'].to_numpy(dtype=float)

rows = []
for k in [20, 40, 80]:
    for shrink in [10, 25, 50]:
        m = BaselineItemKNN(k=k, shrink=shrink, rating_range=reader.rating_scale).fit(tr_part)
        p = np.array([m.predict(u, i) for u, i in zip(val_part['user_id'], val_part['item_id'])])
        rows.append((k, shrink, float(np.sqrt(np.mean((p - val_true) ** 2)))))

tuning = pd.DataFrame(rows, columns=['k', 'shrink', 'val_RMSE']).sort_values('val_RMSE')
display(tuning.head().round(4))

,k,shrink,val_RMSE
8,80,50,0.9182
5,40,50,0.9183
7,80,25,0.9185
4,40,25,0.9188
6,80,10,0.9190


In [18]:
# Топ-10 рекомендацій власним алгоритмом для того самого користувача
own_top10 = own_model.recommend(user_id, n=10)
if own_top10 is not None:
    if movies is not None:
        own_top10 = own_top10.merge(movies, on='item_id', how='left')[['item_id', 'title', 'predicted_rating']]
    display(own_top10.round(3))

,item_id,predicted_rating
0,1414,5.000
1,1417,5.000
2,1294,4.770
3,480,4.738
4,1593,4.710
5,1449,4.706
6,1541,4.662
7,114,4.606
8,479,4.581
9,1236,4.581


**Висновок до додаткового завдання:**
Baseline із зсувами користувача й фільму вже дає значну частину якості, а item-kNN на залишках додає. Обмеження: матриця схожості фільмів `ni × ni` потребує пам’яті O(ni²), алгоритм не працює для нових користувачів (повертається середня оцінка), а розріджені дані роблять схожість ненадійною, що частково компенсує параметр `shrink`.